---
# Copyright (c) 2026 Angela Villota, and collaborators from the CyED block
# Licensed under the PolyForm Noncommercial License 1.0.0.
# Commercial use is prohibited without prior written authorization.
title: Regular expressions in Python
---

[Open this notebook in Google Colab](https://colab.research.google.com/github/angievig/CyED3/blob/main/Week1/S2/regex-python.ipynb)

This notebook turns the ideas from Session 2 into executable experiments. Run each completed example, predict its output before looking, and then modify it. The final section contains guided problems for you to solve.

## Learning goals

By the end, you should be able to:

- choose among `fullmatch`, `match`, `search`, `findall`, and `finditer`;
- build patterns with classes, alternatives, quantifiers, and boundaries;
- extract structured values with groups;
- replace or split text safely; and
- test a pattern with positive, negative, and boundary cases.

## 1. Import the module

Python provides regular expressions through the standard-library `re` module. Patterns should normally be raw strings such as `r"\d+"`.

In [ ]:
import re

print(f"Using Python regex module: {re.__name__}")

## 2. `fullmatch`, `match`, and `search`

These operations answer different questions:

- `fullmatch`: does the **entire string** follow the pattern?
- `match`: does the pattern occur at the **beginning**?
- `search`: does the pattern occur **anywhere**?

Predict the three results before running the cell.

In [ ]:
pattern = r"cat"
text = "catapult has a cat"

results = {
    "fullmatch": re.fullmatch(pattern, text),
    "match": re.match(pattern, text),
    "search": re.search(pattern, text),
}

for operation, result in results.items():
    print(f"{operation:9} ->", result.group() if result else None)

For validation, `fullmatch` is usually the clearest choice.

In [ ]:
binary_pattern = r"[01]+"
examples = ["10110", "10201", "", "000"]

for value in examples:
    valid = re.fullmatch(binary_pattern, value) is not None
    print(f"{value!r:8} -> {valid}")

## 3. Literals, metacharacters, and escaping

Most characters match themselves. Characters such as `.`, `*`, `+`, `?`, `(`, `)`, `[`, `]`, `{`, `}`, `|`, `^`, `$`, and `\` have special meanings.

Use a backslash to match a metacharacter literally. For example, `r"\."` matches a period, while `r"."` matches almost any single character.

In [ ]:
text = "Version 3.1 costs $5"

print("Any character:", re.findall(r".", text)[:8])
print("Literal period:", re.findall(r"\.", text))
print("Literal dollar:", re.findall(r"\$", text))

## 4. Character classes and quantifiers

A character class selects one symbol. A quantifier controls how many times the preceding atom may repeat.

In [ ]:
samples = ["A7", "abc123", "2026", "no-digits", "x_9"]

for value in samples:
    digits = re.findall(r"[0-9]+", value)
    word_chunks = re.findall(r"\w+", value)
    print(f"{value!r:12} digits={digits!s:10} words={word_chunks}")

Common quantifiers are `?` (zero or one), `*` (zero or more), `+` (one or more), and `{n,m}` (between `n` and `m`). Group before quantifying when several symbols must repeat together.

In [ ]:
repeated_block = re.compile(r"(?:ab){2,4}")

for value in ["ab", "abab", "ababab", "abababab", "ababababab"]:
    print(value, "->", bool(re.fullmatch(repeated_block, value)))

`(?:...)` is a **noncapturing group**. It groups the repeated block without adding a captured result.

## 5. Alternatives and boundaries

The pipe `|` selects between alternatives. `^` and `$` describe line boundaries; `\A` and `\Z` describe the boundaries of the entire input. `\b` identifies a word boundary.

In [ ]:
text = "bred red spread red."

print("Without boundaries:", re.findall(r"red", text))
print("Whole word only:", re.findall(r"\bred\b", text))
print("Replacement:", re.sub(r"\bred\b", "brown", text))

Alternatives should be grouped when a surrounding condition applies to all of them.

In [ ]:
weekday_pattern = r"\b(?:Monday|Tuesday|Wednesday|Thursday|Friday)\b"
text = "Monday is busy; Friday is free."

print(re.findall(weekday_pattern, text))
print(re.sub(weekday_pattern, "WEEKDAY", text))

## 6. Finding every match

`findall` returns matched text. Capturing groups change its return shape. `finditer` returns match objects and is better when positions or several groups matter.

In [ ]:
text = "Meeting at 09:45, lunch at 12:30, train at 18:05."
time_pattern = re.compile(r"(?P<hour>[01][0-9]|2[0-3]):(?P<minute>[0-5][0-9])")

print("findall:", time_pattern.findall(text))

for match in time_pattern.finditer(text):
    print({
        "time": match.group(0),
        "hour": match.group("hour"),
        "minute": match.group("minute"),
        "span": match.span(),
    })

## 7. Capturing groups

Parentheses capture parts of a match. Named groups make extracted data easier to understand.

In [ ]:
date_pattern = re.compile(
    r"(?P<year>\d{4})-(?P<month>0[1-9]|1[0-2])-(?P<day>0[1-9]|[12]\d|3[01])"
)

for value in ["2026-07-22", "2026-13-01", "26-07-22"]:
    match = date_pattern.fullmatch(value)
    print(value, "->", match.groupdict() if match else None)

A regex can enforce the displayed format and numeric ranges above, but it does not know calendar rules such as whether April has 30 days. Use `datetime` after regex validation when true calendar validity matters.

## 8. Backreferences

A backreference requires text captured earlier to occur again. In this telephone example, group 1 stores an optional separator and `\1` requires the same separator in the second position.

In [ ]:
phone_pattern = re.compile(r"\d{3}([ -]?)\d{3}\1\d{3}")

phone_cases = [
    ("555555555", True),
    ("555-555-555", True),
    ("555 555 555", True),
    ("555-555 555", False),
    ("555-55-5555", False),
]

for value, expected in phone_cases:
    actual = phone_pattern.fullmatch(value) is not None
    print(f"{value!r:15} expected={expected} actual={actual}")
    assert actual == expected

## 9. Substitution and splitting

`sub` replaces matches. Its optional `count` argument limits the number of replacements. `split` divides text at every match.

In [ ]:
sentences = [
    "They run 5 kilometers in 5 minutes",
    "They were only 5 students",
]

for sentence in sentences:
    print("all:  ", re.sub(r"5", "five", sentence))
    print("first:", re.sub(r"5", "five", sentence, count=1))

In [ ]:
text = "apple, mango;guava | blueberry"
parts = re.split(r"\s*[,;|]\s*", text)
print(parts)

## 10. A reusable testing helper

A pattern is only as convincing as its test cases. This helper reports mismatches without stopping the notebook.

In [ ]:
def check_fullmatch(pattern, cases):
    """Compare full-match results with expected booleans."""
    passed = 0
    for value, expected in cases:
        actual = re.fullmatch(pattern, value) is not None
        mark = "✓" if actual == expected else "✗"
        print(f"{mark} {value!r:20} expected={expected} actual={actual}")
        passed += actual == expected
    print(f"Passed {passed}/{len(cases)} cases")
    return passed == len(cases)

identifier_cases = [
    ("data_2026", True),
    ("A", True),
    ("2data", False),
    ("has-hyphen", False),
    ("", False),
]

check_fullmatch(r"[A-Za-z_]\w*", identifier_cases)

# Guided practice

Complete the next cells by replacing each `TODO_PATTERN`. Run the provided cases, study every failure, and revise the pattern. The starter value `r"(?!)"` deliberately matches nothing but allows the notebook to run before you solve the task.

## Exercise 1 — Hexadecimal address

Find `0xA0` anywhere in each line. This is a search problem, not full-string validation.

In [ ]:
lines = [
    "start address: 0xA0",
    "func address: 0xC0",
    "end address: 0xFF",
    "backup: 0xA0",
]

address_pattern = r"(?!)"  # TODO: replace with your pattern

for line in lines:
    print(line, "->", re.search(address_pattern, line) is not None)

<details><summary>Partial solution</summary>

The required text has no regex metacharacters, so begin with a literal pattern. Use `re.search` because other text may occur before and after it.

</details>

## Exercise 2 — Whole-word replacement

Replace only the complete word `red` with `brown` in `bred red spread credible red.`

In [ ]:
text = "bred red spread credible red."
whole_red_pattern = r"(?!)"  # TODO: add word boundaries
result = re.sub(whole_red_pattern, "brown", text)
print(result)
print("Expected: bred brown spread credible brown.")

## Exercise 3 — Two required letters

Filter words containing both `e` and `n`, in either order.

In [ ]:
words = ["fast", "some", "wonderful", "army", "new", "corn", "energy"]
both_letters_pattern = r"(?!)"  # TODO

matches = [word for word in words if re.search(both_letters_pattern, word)]
print(matches)
print("Expected: ['wonderful', 'new', 'energy']")

<details><summary>Hint</summary>

Use an alternative: one branch for `e` before `n`, and another for `n` before `e`. The text between them can be arbitrary.

</details>

## Exercise 4 — Extract times

Extract the full time, hour, minute, and period (`am` or `pm`) from every sentence. Permit an optional space before the period.

In [ ]:
time_sentences = [
    "The appointment is at 2:45pm.",
    "The dentist is at 11:30 am.",
    "The train leaves at 08:10 am and arrives at 09:00am.",
]

time_with_period_pattern = re.compile(r"(?!)")  # TODO: use named groups

for sentence in time_sentences:
    for match in time_with_period_pattern.finditer(sentence):
        print(match.group(0), match.groupdict())

## Exercise 5 — Validate usernames

Write a pattern for usernames that:

- contain 8 to 16 characters;
- start with a letter;
- contain at least one underscore;
- contain at least one digit;
- end with the same letter with which they started, case-sensitively; and
- contain only letters, digits, and underscores.

Lookaheads can require a digit and underscore without consuming them. A captured first letter and backreference can enforce the ending.

In [ ]:
username_cases = [
    ("User_123U", True),
    ("User_1234", False),
    ("a_user1a", True),
    ("Z_user_9Z", True),
    ("a__1b", False),
    ("_User_1_", False),
]

username_pattern = r"(?!)"  # TODO
check_fullmatch(username_pattern, username_cases)

<details><summary>Partial construction</summary>

Build the pattern in layers:

```python
r"([A-Za-z])"       # capture the first letter
r"(?=...\d)"        # require a digit
r"(?=..._)"         # require an underscore
r"[A-Za-z0-9_]{...}" # control allowed characters and total length
r"\1"               # repeat the first letter
```

Account for the captured first and final letters when choosing the repetition range in the middle.

</details>

## Exercise 6 — Design your own tests

Choose one pattern above and add at least:

1. two ordinary valid cases;
2. two invalid cases that each violate one requirement;
3. the shortest valid case;
4. the longest valid case; and
5. an empty string.

Explain what defect each negative case is intended to detect.

In [ ]:
# Add your test cases here.
my_cases = [
    # ("example", True),
]

# check_fullmatch(your_pattern, my_cases)

## Reflection

Before moving on, answer these questions in a Markdown cell of your own:

1. When did `fullmatch` make a pattern simpler?
2. Which test case exposed the most useful mistake?
3. When would `finditer` be preferable to `findall`?
4. What part of a regex problem should be solved before writing Python code?